# Part 2 — Tool Calling & Reasoning

Pipeline: extract raw date text from pages 1 and 36 (LLM, structured output) -> normalize to ISO
via a deterministic datetime tool (available both as a local MCP tool and as a plain LangChain
`@tool` fallback, both wrapping the same `tools/datetime_core.py` logic) -> classify each date
relative to reference date `2024-01-01` (LLM, structured output).

See `README.md` for the full writeup, including the reasoning bug found and fixed during
development (the classification step initially used the model's own real-world "today"
instead of the given reference date).

## Step 1: MCP tool sanity check

Confirms the datetime tool is correctly registered and callable through the actual MCP
protocol layer, not just importable as a plain function.

In [1]:
import asyncio
from tools.datetime_mcp import mcp

async def check_mcp():
    tools = await mcp.list_tools()
    for t in tools:
        print('registered tool:', t.name, '-', t.description)
    result = await mcp.call_tool('normalize_date', {'raw_text': 'Distributed on Budget Day: 16 February 2024'})
    print('MCP call result:', result.structured_content)

await check_mcp()

registered tool: normalize_date - Normalizes a date found in raw_text to ISO 8601 (YYYY-MM-DD).
MCP call result: {'result': '2024-02-16'}


## Step 2: full pipeline (extraction -> normalization -> classification)

Uses the fallback `@tool` path internally (works standalone, no MCP client needed) for
the normalization step, since a full pipeline shouldn't require spinning up a separate MCP
server process — the MCP server above is verified independently as the assignment requires.

In [2]:
from part2_pipeline import run_pipeline
from llm_config import HAIKU_MODEL

result = run_pipeline(model=HAIKU_MODEL, max_tokens=1024)
print(result.model_dump_json(indent=2))

HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


{
  "dates": [
    {
      "original_text": "Distributed on Budget Day: 16 February 2024",
      "normalized_date": "2024-02-16",
      "status": "Upcoming"
    },
    {
      "original_text": "Estate Duty does not apply to a person who dies after 15 February 2008.",
      "normalized_date": "2008-02-15",
      "status": "Expired"
    }
  ]
}


## Verification against ground truth

- `2024-02-16` (document distribution date) must be **Upcoming** relative to `2024-01-01` — this
  is literally the assignment's own sample output example.
- `2008-02-15` (estate duty cutoff) is classified **Expired** per our documented assumption
  (see README) — this one is a genuine judgment call, not a clear-cut case.

In [3]:
ground_truth = {
    "2024-02-16": "Upcoming",
    "2008-02-15": "Expired",
}

print(f"{'normalized_date':<18}{'status':<12}{'expected':<12}{'match'}")
for d in result.dates:
    expected = ground_truth.get(d.normalized_date, "(no ground truth)")
    match = d.status == expected
    print(f"{d.normalized_date:<18}{d.status:<12}{expected:<12}{'PASS' if match else 'FAIL'}")

normalized_date   status      expected    match
2024-02-16        Upcoming    Upcoming    PASS
2008-02-15        Expired     Expired     PASS
